# VorFlow Smoothing Example Using Public Ontario Data for KW
The purpose of this script (Jupyter Notebook) is to re-build the prototype a pre-conditioner for Vorflow using publically available data for Kitchener-Waterloo area. The notebook is organized in sections that can be reviewed by clikcing the Table of Contents button on the left.

# Initial Setup


## Import Dependencies

In [ ]:
import os, sys, math
import pandas as pd
import numpy as np
import seaborn as sns
import geopandas as gpd 
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LogNorm, Normalize
from IPython.display import display,HTML

import rasterio
from rasterio.crs import CRS
import contextily as ctx
import shapely.vectorized
from shapely.ops import unary_union
from shapely.geometry import box, Point, Polygon, LineString
from pyproj import CRS

import flopy
import flopy.utils as fpu
import flopy.utils.binaryfile as bf
from flopy.utils import Raster, CellBudgetFile
from flopy.utils.gridintersect import GridIntersect
from flopy.plot import PlotCrossSection, PlotMapView

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # for shapely and flopy
warnings.filterwarnings("ignore", category=ResourceWarning)     # for flopy

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator


## Notebook Management [optional]
The cell below will create a backup of the parent environment that ran this Notebook. Running this is optionaL.

In [ ]:
# Local/Custom Packages and Classes
sys.path.append(r'C:\Repo')
from hydro_tools.utils import backup_env, notebook_tools

backup_env(True)

## Input Variables
Define the model name, input files and workspace

In [ ]:
# MODEL BASICS
modelname = "KW"
workspace = r'.\KW\model\flopy'

# Griding Options
boundary_resolution = 500
inner_resolution    = 50
pt_resolution       = 10
growth_factor       = 1.2
background_lc       = 1000
lloyd_iterations    = 10

# INPUT FILES
domain_file        = r'.\KW\GIS\Domain.shp'                                                 # rectangular extents of the model
active_domain_file = r'.\KW\GIS\OHN_watershed.shp'                                          # extents of the model
well_file          = r'.\KW\GIS\PTTW.shp'                                                   # point shapefile with all obs and pw wells (WEL input)

sw_lines           = r'.\KW\GIS\OHN_watercourse_s20.shp'
sw_polys           = r'.\KW\GIS\OHN_waterbodies_mod.shp'

# SPATIAL INFO
epsg           = 26917                     # coordinate reference system
crs            = f'EPSG:{epsg}'            

# BASEMAP GENERAL SETTINGS
image_url         = 'https://ws.lioservices.lrc.gov.on.ca/arcgis2/rest/services/LIO_Imagery/Ontario_Imagery_Web_Map_Service/MapServer/tile/{z}/{y}/{x}'
alpha= 0.4
zoom=12

# Visualization Settings
plot_inputs       = True         # Turn on/off extra plots
plot_outputs      = True         # Turn on/off extra plots
fullplotsize      = 30           # size of single plots
layerplotsize     = 20           # size of layered plots


## Functions

In [ ]:
def gdf_to_shp(gdf, workspace, filename, exclude_cols=None, fmt="GPKG"):
    filename = os.path.join(workspace,filename)
    
    if fmt.upper()=="SHP":
        fmt="ESRI Shapefile"
    elif fmt.upper()=="GEOJSON":
        fmt="GeoJSON"
    else:
        fmt="GPKG"
    if exclude_cols is not None:
        cols_to_keep = [c for c in gdf.columns if c not in exclude_cols]
    else:
        cols_to_keep = gdf.columns
    try:
        gdf[cols_to_keep].to_file(filename, driver=fmt)
        print(f'EXPORT: {filename}')
    except:
        print(f'ERROR: {filename} is locked')


## Load In Vector Inputs

In [ ]:
domain = gpd.read_file(domain_file)
well_id = 'PermitNo'
active_domain = gpd.read_file(active_domain_file)
wells = gpd.read_file(well_file)
wells = wells[wells["ACTIVE"]=='Yes'].clip(active_domain).reset_index(drop=True)
display(HTML(wells.head().to_html(render_links=True, escape=False)))

# Plot
if plot_inputs:
    fig, ax = plt.subplots(figsize=(20, 20))
    wells.plot(ax=ax, markersize=3, color='purple', zorder=5)
    #for idx, row in wells.iterrows():
    #    x, y = row.geometry.x, row.geometry.y
    #    ax.text(x, y-2000, row["WELL_ID"], fontsize=8, ha="center", va="bottom", color='purple')
    
    domain.plot(ax=ax, facecolor='none', edgecolor='black', lw=1, zorder=3)
    active_domain.plot(ax=ax, facecolor='none', edgecolor='black', lw=2, zorder=3)
    try:
        ctx.add_basemap(ax, source=image_url, zoom=zoom, crs=f'EPSG:{epsg}',alpha=alpha)
    except:
        print("Can't reach basemap")

    ax.set_title("Inputs")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.show()


# Grid Builder

## Load and Process BC Shapes

In [ ]:
# Get the SW Features
print('Reading Shapefiles                                        ', end = f'\r')
sw_poly_gdf = gpd.read_file(sw_polys)
sw_line_gdf = gpd.read_file(sw_lines)

# Filter features where type is one of the desired boundary condition types "RIV", "DRN"
sw_poly_gdf = sw_poly_gdf[sw_poly_gdf["Type"].isin(["RIV","DRN"])].clip(active_domain).reset_index(drop=True)
sw_line_gdf = sw_line_gdf.clip(active_domain).reset_index(drop=True)

print('Processing Shapefiles                                      ', end = f'\r')
# Surface water outside of a surface water body (pond/lake/wide riv)
sw_line_gdf = gpd.overlay(sw_line_gdf, sw_poly_gdf, how='difference') 

# Get SW poly boundary instead of polygon
sw_arcs_gdf = sw_poly_gdf.copy()
sw_arcs_gdf['geometry'] = sw_poly_gdf.boundary

if plot_inputs:
    fig, ax = plt.subplots(figsize=(20, 20))
    wells.plot(ax=ax, color='purple', zorder=5)
    # Label wells
    #for idx, row in wells.iterrows():
    #    x, y = row.geometry.x, row.geometry.y
    #    ax.text(x, y-400, row["WELL_ID"], fontsize=8, ha="center", va="bottom", color='purple')

    sw_poly_gdf.plot(ax=ax, color='blue', facecolor='lightblue', lw=1)
    sw_line_gdf.plot(ax=ax, color='blue', lw=1)
    
    active_domain.plot(ax=ax, facecolor='none', edgecolor='black', lw=2, zorder=3)
    try:
        ctx.add_basemap(ax, source=image_url, zoom=zoom, crs=f'EPSG:{epsg}',alpha=alpha)
    except:
        print("Can't reach basemap")

    ax.set_title("Inputs")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.show()


## Grid Generation using VorFlow

In [ ]:

blueprint = ConceptualMesh(crs=crs)
blueprint.add_polygon(active_domain.geometry[0], zone_id=1)

boundary = active_domain.copy()
boundary['geometry'] = active_domain.boundary
blueprint.add_line(boundary.geometry[0], resolution = boundary_resolution, growth_factor=growth_factor, line_id="Boundary")

for _, rec in wells.iterrows():
    #print(w.geometry,w.Boundname)
    blueprint.add_point(rec.geometry, point_id=rec.PermitNo, resolution=pt_resolution, growth_factor=growth_factor)

for idx, rec in sw_line_gdf.iterrows():
    print(idx,"                         ", end='\r')
    blueprint.add_line(rec.geometry, line_id=f'sw_{idx}', resolution=inner_resolution, growth_factor=growth_factor)
fid = idx

for idx, rec in sw_arcs_gdf.iterrows():
    shp_id = idx + fid
    print(shp_id,"                         ", end='\r')
    blueprint.add_line(rec.geometry, line_id=f'sw_{shp_id}', resolution=inner_resolution, growth_factor=growth_factor, is_barrier=True,)
fid = shp_id


clean_polys, clean_lines, clean_pts = blueprint.generate()

mesher = MeshGenerator(background_lc=background_lc)
mesher.generate(clean_polys, clean_lines, clean_pts)

tessellator = VoronoiTessellator(mesher, blueprint, clip_to_boundary=True, lloyd_iterations=lloyd_iterations)
gdf_voronoi = tessellator.generate()
print(tessellator.lloyd_report)


## Plot Unstructured Grid Output

In [ ]:
outpath = os.path.join(workspace, "outputs")
gdf_to_shp(gdf_voronoi,   outpath, f"{modelname}_50_voronoi.gpkg", exclude_cols=['pts','polygons'], fmt="GPKG")
#gdf_voronoi.to_file(os.path.join(outpath,f"{modelname}_voronoi.gpkg"), driver = 'GPKG')

if plot_inputs:
    fig, ax = plt.subplots(figsize=(20, 20))
    wells.plot(ax=ax, markersize=3, color='purple', zorder=5)
    #for idx, row in wells.iterrows():
    #    x, y = row.geometry.x, row.geometry.y
    #    ax.text(x, y-400, row["boundname"], fontsize=9, ha="center", va="bottom", color='purple')

    gdf_voronoi.plot(ax=ax, facecolor='none', edgecolor='black', lw=1, zorder=3)
    #active_domain.plot(ax=ax, facecolor='none', edgecolor='black', lw=2, zorder=3)
    try:
        ctx.add_basemap(ax, source=image_url, zoom=zoom, crs=f'EPSG:{epsg}',alpha=alpha)
    except:
        print("Can't reach basemap")

    ax.set_title("Inputs")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.show()
